# Alzheimer's MRI Classification - ResNet50 with Grad-CAM

Classifies brain MRI scans into 4 Alzheimer's stages:
- **NonDemented** - Healthy control
- **VeryMildDemented** - Very mild cognitive impairment
- **MildDemented** - Mild Alzheimer's
- **ModerateDemented** - Moderate Alzheimer's

## 1. Setup

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, classification_report

warnings.simplefilter('ignore')
tf.random.set_seed(54)
np.random.seed(54)
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Mount Drive & Extract Dataset

In [ ]:
from google.colab import drive
import zipfile

drive.mount('/content/drive', force_remount=True)

zip_path = '/content/drive/MyDrive/archive (3).zip'  # UPDATE THIS PATH
extract_base = '/content/DATASEt'

if not os.path.exists(extract_base) or not os.listdir(extract_base):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_base)
    print(f'Extracted to {extract_base}')
else:
    print('Already extracted')

actual_root = extract_base
contents = os.listdir(extract_base)
if len(contents) == 1 and os.path.isdir(os.path.join(extract_base, contents[0])):
    actual_root = os.path.join(extract_base, contents[0])

TRAIN_DIR = os.path.join(actual_root, 'AugmentedAlzheimerDataset')
TEST_DIR = os.path.join(actual_root, 'OriginalDataset')
print(f'Train: {TRAIN_DIR}')
print(f'Test: {TEST_DIR}')

## 3. Load Datasets

In [ ]:
IMG_SIZE = (176, 208)
BATCH_SIZE = 8
AUTOTUNE = tf.data.experimental.AUTOTUNE
SEED = 54
CLASS_NAMES = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
NUM_CLASSES = len(CLASS_NAMES)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

train_ds.class_names = CLASS_NAMES
val_ds.class_names = CLASS_NAMES
test_ds.class_names = CLASS_NAMES

## 4. Class Weights

In [ ]:
all_labels = []
for _, labels in train_ds:
    all_labels.extend(labels.numpy())

weights = class_weight.compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weights = {i: w for i, w in enumerate(weights)}
print('Class weights:')
for idx, w in class_weights.items():
    print(f'  {CLASS_NAMES[idx]}: {w:.4f}')

## 5. Prepare Datasets

In [ ]:
def one_hot_label(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_pref = train_ds.map(one_hot_label, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_pref = val_ds.map(one_hot_label, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_pref = test_ds.map(one_hot_label, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

for sample_images, sample_labels in test_pref.take(1):
    break
print('Datasets ready')

## 6. Build & Train ResNet50

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications.resnet50 import ResNet50

def build_resnet50():
    base = ResNet50(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), weights='imagenet', include_top=False)
    base.trainable = False
    x = layers.Flatten()(base.output)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs=base.input, outputs=out)

model = build_resnet50()
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=[tf.keras.metrics.AUC(name='auc'), 'acc'])
model.summary()

In [ ]:
EPOCHS = 10

callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(train_pref, epochs=EPOCHS, validation_data=val_pref,
                    callbacks=callback, class_weight=class_weights)

model.save('resnet_normalized.h5')
print('Model saved!')

## 7. Training Curves

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['acc'], label='Train', marker='o')
plt.plot(history.history['val_acc'], label='Val', marker='o')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train', marker='o')
plt.plot(history.history['val_loss'], label='Val', marker='o')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

## 8. Evaluation

In [ ]:
y_true, y_pred = [], []
for imgs, lbls in test_pref:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(lbls.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true, y_pred = np.array(y_true), np.array(y_pred)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix'); plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.show()

## 9. Grad-CAM Explainability

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name='conv5_block3_out'):
    grad_model = tf.keras.models.Model([model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = last_conv_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap))
    return heatmap.numpy()

def apply_gradcam(img, heatmap, alpha=0.4):
    img = img.astype('uint8')
    heatmap = np.uint8(255 * heatmap)
    jet = plt.get_cmap('jet')
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)
    return tf.keras.preprocessing.image.array_to_img(jet_heatmap * alpha + img)

In [ ]:
plt.figure(figsize=(15, 10))
for i in range(min(4, len(sample_images))):
    img_tensor = tf.expand_dims(sample_images[i], 0)
    heatmap = make_gradcam_heatmap(img_tensor, model)
    cam_img = apply_gradcam(sample_images[i].numpy(), heatmap)

    plt.subplot(2, 4, i*2+1)
    plt.imshow(sample_images[i].numpy().astype('uint8'))
    plt.title(f'Original: {CLASS_NAMES[np.argmax(sample_labels[i])]}')
    plt.axis('off')

    plt.subplot(2, 4, i*2+2)
    plt.imshow(cam_img)
    plt.title('Grad-CAM')
    plt.axis('off')
plt.tight_layout(); plt.show()

## 10. Save Model to Drive & Launch Gradio App

In [ ]:
import shutil
shutil.copy('resnet_normalized.h5', '/content/drive/MyDrive/resnet_normalized.h5')
print('Model copied to Google Drive!')

In [ ]:
!pip install gradio -q
import gradio as gr

def full_analysis(image):
    if image is None: return None, 'No image', None
    img_resized = tf.image.resize(image, IMG_SIZE)
    img_batch = tf.expand_dims(img_resized, axis=0)
    preds = model.predict(img_batch)
    idx = np.argmax(preds[0])
    label = CLASS_NAMES[idx]
    conf = preds[0][idx] * 100
    prob_dict = {CLASS_NAMES[i]: float(preds[0][i]) for i in range(NUM_CLASSES)}
    heatmap = make_gradcam_heatmap(img_batch, model)
    cam_img = apply_gradcam(image.numpy().astype('uint8'), heatmap)
    result = f'### Finding: **{label}**\n**Confidence:** {conf:.2f}%'
    return cam_img, result, prob_dict

with gr.Blocks(title="Alzheimer's AI") as iface:
    gr.Markdown('# Brain MRI Alzheimer\'s Diagnostic')
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type='numpy', label='Upload MRI')
            submit_btn = gr.Button('Run Analysis', variant='primary')
        with gr.Column():
            image_output = gr.Image(label='Grad-CAM')
    with gr.Row():
        text_output = gr.Markdown()
        label_output = gr.Label(num_top_classes=4)
    submit_btn.click(fn=full_analysis, inputs=image_input,
                     outputs=[image_output, text_output, label_output])

iface.launch(share=True)